# Phase 2 — Multi-Object Tracking
## ByteTrack 직접 구현 + MOT 평가 파이프라인

### 학습 목표
- **Kalman Filter** 직접 구현 — 위치 예측의 수학적 원리 이해
- **Hungarian Algorithm** 직접 구현 — IoU 기반 매칭 원리
- **ByteTrack 알고리즘** 구현 — High/Low score 분리 매칭 전략
- **MOTA / HOTA 평가** — Detection mAP와 다른 Tracking 지표 이해
- **YOLOv8 + ByteTrack** 연동 — Phase 1 결과물 재활용

### Phase 1 → Phase 2 연결
```
IoU 직접 구현   → 매칭 비용 행렬 계산에 그대로 사용
NMS             → 중복 track 제거 동일 원리
YOLOv8 추론     → ByteTrack의 detector로 사용
FP/FN 분석      → MOTA = 1 - (FP+FN+ID_SW)/GT 로 확장
```


## 0. 환경 확인


In [ ]:
import torch
import numpy as np
import cv2
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')
print(f'OpenCV: {cv2.__version__}')


## 1. Kalman Filter — 위치 예측의 핵심

### 왜 Kalman Filter인가?
- 카메라 프레임 사이에 객체가 움직임 → 다음 프레임에서 **어디 있을지 예측** 필요
- 측정값(detection)에는 노이즈가 있음 → 예측 + 측정을 **최적으로 융합**
- Tracking의 모든 알고리즘(SORT, DeepSORT, ByteTrack)이 Kalman Filter 사용

### 상태 벡터 (8차원)
```
x = [cx, cy, s, r, vx, vy, vs, vr]
     위치(4)         속도(4)

cx, cy : 박스 중심 좌표
s      : 박스 면적 (scale)
r      : 종횡비 (aspect ratio, 고정 가정)
vx, vy : cx, cy 속도
vs     : s 속도
vr     : r 속도 (≈ 0)
```

### 핵심 수식
```
예측 단계 (Predict):
  x̂ₖ = F · xₖ₋₁          (상태 전이)
  Pₖ = F · Pₖ₋₁ · Fᵀ + Q  (공분산 전파)

갱신 단계 (Update):
  K  = Pₖ · Hᵀ · (H · Pₖ · Hᵀ + R)⁻¹  (Kalman Gain)
  xₖ = x̂ₖ + K · (z - H · x̂ₖ)          (상태 보정)
  Pₖ = (I - K · H) · Pₖ                 (공분산 보정)
```


In [ ]:
class KalmanBoxTracker:
    """
    단일 객체를 추적하는 Kalman Filter.
    상태: [cx, cy, s, r, vx, vy, vs, vr]  (s=면적, r=종횡비)
    측정: [cx, cy, s, r]
    """
    count = 0  # 전역 track ID 카운터

    def __init__(self, bbox_xyxy):
        """
        bbox_xyxy: [x1, y1, x2, y2]
        """
        # 상태 전이 행렬 F (등속 운동 모델)
        self.F = np.eye(8)
        for i in range(4):
            self.F[i, i+4] = 1.0  # 위치 += 속도

        # 측정 행렬 H (상태에서 측정값 추출)
        self.H = np.zeros((4, 8))
        self.H[:4, :4] = np.eye(4)

        # 측정 노이즈 공분산 R
        self.R = np.eye(4)
        self.R[2, 2] = 10.0  # 면적은 노이즈 크게
        self.R[3, 3] = 10.0

        # 프로세스 노이즈 공분산 Q
        self.Q = np.eye(8)
        self.Q[4:, 4:] *= 0.01  # 속도 노이즈는 작게
        self.Q[6, 6] = 0.0001

        # 초기 상태 공분산 P
        self.P = np.eye(8) * 10.0
        self.P[4:, 4:] *= 1000.0  # 초기 속도 불확실성 크게

        # 초기 상태 x
        cx, cy, s, r = self._xyxy_to_cxcysr(bbox_xyxy)
        self.x = np.array([cx, cy, s, r, 0, 0, 0, 0], dtype=float)

        self.id = KalmanBoxTracker.count
        KalmanBoxTracker.count += 1
        self.age = 0           # 생성 후 총 프레임 수
        self.hits = 0          # 매칭된 횟수
        self.hit_streak = 0    # 연속 매칭 횟수
        self.time_since_update = 0  # 마지막 업데이트 후 프레임 수

    # ── 좌표 변환 ──
    @staticmethod
    def _xyxy_to_cxcysr(bbox):
        x1, y1, x2, y2 = bbox
        w, h = x2 - x1, y2 - y1
        cx, cy = x1 + w/2, y1 + h/2
        s = w * h          # 면적
        r = w / (h + 1e-6) # 종횡비
        return cx, cy, s, r

    def _state_to_xyxy(self):
        cx, cy, s, r = self.x[:4]
        w = np.sqrt(max(s * r, 1e-6))
        h = max(s / (w + 1e-6), 1e-6)
        return [cx-w/2, cy-h/2, cx+w/2, cy+h/2]

    # ── Kalman 예측 ──
    def predict(self):
        """다음 프레임 위치 예측"""
        if self.time_since_update > 0:
            self.hit_streak = 0
        self.time_since_update += 1
        self.age += 1

        # x̂ = F · x
        self.x = self.F @ self.x
        # P = F · P · Fᵀ + Q
        self.P = self.F @ self.P @ self.F.T + self.Q
        return self._state_to_xyxy()

    # ── Kalman 갱신 ──
    def update(self, bbox_xyxy):
        """detection으로 상태 보정"""
        self.time_since_update = 0
        self.hits += 1
        self.hit_streak += 1

        z = np.array(self._xyxy_to_cxcysr(bbox_xyxy))

        # 혁신(innovation): 측정 - 예측
        y = z - self.H @ self.x
        # 혁신 공분산: S = H·P·Hᵀ + R
        S = self.H @ self.P @ self.H.T + self.R
        # Kalman Gain: K = P·Hᵀ·S⁻¹
        K = self.P @ self.H.T @ np.linalg.inv(S)
        # 상태 보정: x = x + K·y
        self.x = self.x + K @ y
        # 공분산 보정: P = (I - K·H)·P
        self.P = (np.eye(8) - K @ self.H) @ self.P

    def get_state(self):
        return self._state_to_xyxy()


# ── 동작 확인 ──
KalmanBoxTracker.count = 0
tracker = KalmanBoxTracker([100, 200, 200, 350])  # [x1,y1,x2,y2]

print('Kalman Filter 동작 확인')
print(f'초기 상태: {[round(v,1) for v in tracker.get_state()]}')

# 3프레임 예측 (detection 없음)
for i in range(3):
    pred = tracker.predict()
    print(f'  predict [{i+1}]: {[round(v,1) for v in pred]}')

# detection으로 갱신
tracker.update([105, 205, 205, 355])
print(f'update 후: {[round(v,1) for v in tracker.get_state()]}')


In [ ]:
# ── Kalman Filter 예측 vs 실제 궤적 시각화 ──
KalmanBoxTracker.count = 0

# 시뮬레이션: 박스가 오른쪽으로 이동
true_positions = [[100 + i*10, 200, 200 + i*10, 350] for i in range(15)]

tracker = KalmanBoxTracker(true_positions[0])
predicted, updated = [], []

for i, bbox in enumerate(true_positions):
    # 노이즈 추가
    noisy = [v + np.random.normal(0, 3) for v in bbox]

    pred = tracker.predict()
    predicted.append([(pred[0]+pred[2])/2, (pred[1]+pred[3])/2])  # 중심

    tracker.update(noisy)
    state = tracker.get_state()
    updated.append([(state[0]+state[2])/2, (state[1]+state[3])/2])

true_cx   = [(b[0]+b[2])/2 for b in true_positions]
pred_cx   = [p[0] for p in predicted]
update_cx = [u[0] for u in updated]
frames    = list(range(len(true_positions)))

plt.figure(figsize=(10, 4))
plt.plot(frames, true_cx,   'g-o', label='실제 위치 (GT)', linewidth=2)
plt.plot(frames, pred_cx,   'r--s', label='Kalman 예측', linewidth=2)
plt.plot(frames, update_cx, 'b-^', label='Kalman 보정 후', linewidth=2)
plt.xlabel('프레임')
plt.ylabel('중심 x 좌표')
plt.title('Kalman Filter — 예측 vs 보정 궤적')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('kalman_trajectory.png', dpi=150, bbox_inches='tight')
plt.show()
print('Kalman Filter: 측정 노이즈에도 불구하고 실제 궤적에 수렴')


## 2. Hungarian Algorithm — 최적 매칭

### 문제 정의
```
N개의 기존 track  ↔  M개의 새 detection
→ 비용(cost)이 최소가 되도록 1:1 매칭
```

### 비용 행렬
- `cost[i][j] = 1 - IoU(track_i_pred, detection_j)`
- IoU가 높을수록(유사할수록) 비용이 낮음 → 매칭 우선

### scipy.optimize.linear_sum_assignment
- Kuhn-Munkres 알고리즘 구현체
- O(N³) 시간복잡도
- 반환: (row_indices, col_indices) = 최적 매칭 쌍


In [ ]:
from scipy.optimize import linear_sum_assignment


def compute_iou_matrix(boxes_a, boxes_b):
    """
    boxes_a: (N, 4) tracks 예측 박스
    boxes_b: (M, 4) detections
    반환: (N, M) IoU 행렬
    """
    iou_mat = np.zeros((len(boxes_a), len(boxes_b)))
    for i, a in enumerate(boxes_a):
        for j, b in enumerate(boxes_b):
            # Phase 1에서 구현한 compute_iou 재사용
            xi1 = max(a[0], b[0]);  yi1 = max(a[1], b[1])
            xi2 = min(a[2], b[2]);  yi2 = min(a[3], b[3])
            inter = max(0, xi2-xi1) * max(0, yi2-yi1)
            area_a = (a[2]-a[0]) * (a[3]-a[1])
            area_b = (b[2]-b[0]) * (b[3]-b[1])
            union = area_a + area_b - inter
            iou_mat[i, j] = inter / (union + 1e-6)
    return iou_mat


def hungarian_match(tracks_pred, detections, iou_threshold=0.3):
    """
    Hungarian Algorithm으로 track-detection 매칭.
    반환:
        matched      : [(track_idx, det_idx), ...]
        unmatched_trk: [track_idx, ...]  (매칭 안 된 track → 소멸 후보)
        unmatched_det: [det_idx, ...]   (매칭 안 된 detection → 새 track)
    """
    if len(tracks_pred) == 0:
        return [], [], list(range(len(detections)))
    if len(detections) == 0:
        return [], list(range(len(tracks_pred))), []

    iou_mat  = compute_iou_matrix(tracks_pred, detections)
    cost_mat = 1 - iou_mat  # 비용 = 1 - IoU

    row_ind, col_ind = linear_sum_assignment(cost_mat)

    matched, unmatched_trk, unmatched_det = [], [], []

    matched_trk = set(row_ind)
    matched_det = set(col_ind)

    for r, c in zip(row_ind, col_ind):
        if iou_mat[r, c] >= iou_threshold:
            matched.append((r, c))
        else:  # IoU 너무 낮으면 매칭 거부
            unmatched_trk.append(r)
            unmatched_det.append(c)

    for i in range(len(tracks_pred)):
        if i not in matched_trk:
            unmatched_trk.append(i)
    for j in range(len(detections)):
        if j not in matched_det:
            unmatched_det.append(j)

    return matched, unmatched_trk, unmatched_det


# ── 동작 확인 ──
tracks = [[100,200,200,350], [300,100,400,250], [500,300,600,450]]
dets   = [[105,205,205,355], [310,105,410,255], [700,400,800,550]]  # 세 번째는 새 객체

matched, unmatched_trk, unmatched_det = hungarian_match(tracks, dets)
print('Hungarian Matching 결과')
print(f'  매칭:          {matched}        → (track_idx, det_idx)')
print(f'  미매칭 track:  {unmatched_trk}  → 소멸 후보')
print(f'  미매칭 det:    {unmatched_det}  → 새 track 생성')


## 3. ByteTrack 알고리즘

### SORT vs ByteTrack
| | SORT (2016) | ByteTrack (2022) |
|---|---|---|
| 매칭 대상 | high conf detection만 | **high + low conf 모두** |
| 매칭 단계 | 1단계 | **2단계** |
| ID Switch | 많음 | 적음 |
| 가려진 객체 | 유실됨 | **low conf로 유지** |

### ByteTrack 2단계 매칭
```
1단계: 활성 track ↔ high score detection (conf ≥ thresh_high)
       → 확실한 객체들끼리 먼저 매칭

2단계: 미매칭 track ↔ low score detection (thresh_low ≤ conf < thresh_high)
       → 가려지거나 흐릿한 객체도 기존 track과 연결
       → FP 방지: 아예 새로운 track은 생성하지 않음
```


In [ ]:
class ByteTracker:
    """
    ByteTrack 구현 (Zhang et al., 2022)
    논문: https://arxiv.org/abs/2110.06864

    핵심 아이디어: low confidence detection을 버리지 않고
    2단계 매칭에서 기존 track 유지에 활용 → ID Switch 감소
    """
    def __init__(self,
                 thresh_high=0.6,   # 1단계 매칭 conf 임계값
                 thresh_low=0.1,    # 2단계 매칭 최소 conf
                 max_age=30,        # 미매칭 허용 프레임 수
                 min_hits=3,        # track 확정 최소 연속 매칭 수
                 iou_threshold=0.3):
        self.thresh_high   = thresh_high
        self.thresh_low    = thresh_low
        self.max_age       = max_age
        self.min_hits      = min_hits
        self.iou_threshold = iou_threshold

        self.trackers   = []  # 활성 KalmanBoxTracker 목록
        self.frame_count = 0
        KalmanBoxTracker.count = 0

    def update(self, detections):
        """
        detections: list of [x1, y1, x2, y2, score]
        반환: list of [x1, y1, x2, y2, track_id]
        """
        self.frame_count += 1

        # ── 예측 단계: 모든 tracker 1프레임 전진 ──
        pred_boxes = []
        for trk in self.trackers:
            pred_boxes.append(trk.predict())

        # ── detection 분리 ──
        dets = np.array(detections) if len(detections) else np.zeros((0, 5))
        high_mask = dets[:, 4] >= self.thresh_high  if len(dets) else np.array([], bool)
        low_mask  = (dets[:, 4] >= self.thresh_low) & ~high_mask if len(dets) else np.array([], bool)
        high_dets = dets[high_mask]
        low_dets  = dets[low_mask]

        # ── 1단계: 활성 track ↔ high score detection ──
        matched1, unmatched_trk1, unmatched_det_high = hungarian_match(
            pred_boxes, high_dets[:, :4].tolist() if len(high_dets) else [],
            self.iou_threshold
        )
        for ti, di in matched1:
            self.trackers[ti].update(high_dets[di, :4])

        # ── 2단계: 미매칭 track ↔ low score detection ──
        unmatched_trackers = [self.trackers[i] for i in unmatched_trk1]
        unmatched_pred     = [pred_boxes[i] for i in unmatched_trk1]

        matched2, unmatched_trk2_idx, _ = hungarian_match(
            unmatched_pred, low_dets[:, :4].tolist() if len(low_dets) else [],
            self.iou_threshold
        )
        for ti, di in matched2:
            unmatched_trackers[ti].update(low_dets[di, :4])

        # ── 미매칭 track 처리: max_age 초과 시 삭제 ──
        confirmed_unmatched_trk = set(unmatched_trk1)
        for i in unmatched_trk2_idx:
            confirmed_unmatched_trk.add(unmatched_trk1[i])

        self.trackers = [
            t for i, t in enumerate(self.trackers)
            if not (i in confirmed_unmatched_trk and t.time_since_update > self.max_age)
        ]

        # ── 새 track 생성: 미매칭 high detection만 ──
        for i in unmatched_det_high:
            self.trackers.append(KalmanBoxTracker(high_dets[i, :4]))

        # ── 결과 반환: min_hits 이상 매칭된 확정 track ──
        results = []
        for trk in self.trackers:
            if (trk.hit_streak >= self.min_hits or self.frame_count <= self.min_hits) \
               and trk.time_since_update <= 1:
                box = trk.get_state()
                results.append([*box, trk.id])
        return results


print('ByteTracker 클래스 정의 완료')


## 4. 시뮬레이션 — ByteTrack 동작 검증

실제 비디오 없이 가상 시나리오로 알고리즘 검증:
- 객체 A: 프레임 내내 등속 이동
- 객체 B: 중간에 가려짐(occlusion) 후 재등장
- 노이즈: 각 프레임 detection에 랜덤 오차


In [ ]:
def simulate_tracking(n_frames=30, noise=5.0):
    """가상 시나리오로 ByteTrack 동작 검증"""
    np.random.seed(42)
    tracker = ByteTracker(thresh_high=0.5, thresh_low=0.1, max_age=5, min_hits=2)

    all_results = []  # 프레임별 track 결과
    gt_tracks = {}

    for f in range(n_frames):
        dets = []

        # 객체 A: 왼→오른쪽 이동 (항상 존재, high conf)
        ax = 50 + f * 12
        dets.append([
            ax + np.random.normal(0, noise), 100 + np.random.normal(0, noise),
            ax + 80 + np.random.normal(0, noise), 200 + np.random.normal(0, noise),
            0.9
        ])
        gt_tracks.setdefault(0, []).append(ax + 40)

        # 객체 B: 프레임 5~25 존재, 14~18은 low conf (가려짐)
        if 5 <= f <= 25:
            bx = 400 - f * 5
            conf = 0.2 if 14 <= f <= 18 else 0.85  # occlusion 구간
            dets.append([
                bx + np.random.normal(0, noise), 300 + np.random.normal(0, noise),
                bx + 70 + np.random.normal(0, noise), 400 + np.random.normal(0, noise),
                conf
            ])
            gt_tracks.setdefault(1, []).append((f - 5, bx + 35))

        results = tracker.update(dets)
        all_results.append((f, results))

    return all_results, gt_tracks


results, gt_tracks = simulate_tracking()

# Track ID 통계
id_history = {}
for f, tracks in results:
    for t in tracks:
        tid = int(t[4])
        id_history.setdefault(tid, []).append(f)

print('ByteTrack 시뮬레이션 결과')
print(f'생성된 Track 수: {len(id_history)}')
for tid, frames in sorted(id_history.items()):
    print(f'  Track {tid}: 프레임 {min(frames)}~{max(frames)} ({len(frames)}프레임 유지)')


In [ ]:
# ── 궤적 시각화 ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 왼쪽: Track 궤적 (중심 x 좌표)
colors = plt.cm.tab10.colors
for i, (tid, frames) in enumerate(sorted(id_history.items())):
    cx_list = []
    for f, tracks in results:
        if f in frames:
            for t in tracks:
                if int(t[4]) == tid:
                    cx_list.append(((t[0]+t[2])/2))
    axes[0].plot(frames, cx_list, 'o-', color=colors[i % 10],
                 label=f'Track {tid}', linewidth=2, markersize=4)

axes[0].set_title('ByteTrack 추적 궤적 (중심 x)', fontsize=12)
axes[0].set_xlabel('프레임')
axes[0].set_ylabel('x 좌표')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].axvspan(14, 18, alpha=0.15, color='red', label='occlusion 구간')

# 오른쪽: 프레임별 활성 track 수
n_active = [len(t) for _, t in results]
axes[1].bar(range(len(n_active)), n_active, color='steelblue', alpha=0.8)
axes[1].axvspan(14, 18, alpha=0.15, color='red')
axes[1].set_title('프레임별 활성 Track 수', fontsize=12)
axes[1].set_xlabel('프레임')
axes[1].set_ylabel('Track 수')
axes[1].set_ylim(0, 4)
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle('ByteTrack 시뮬레이션 — occlusion 구간(빨간 음영)에서 Track 유지 여부', fontsize=13)
plt.tight_layout()
plt.savefig('bytetrack_simulation.png', dpi=150, bbox_inches='tight')
plt.show()
print('occlusion 구간(14~18프레임)에서 객체 B의 Track ID가 유지되면 성공')


## 5. YOLOv8 + ByteTrack 연동

Phase 1에서 학습한 YOLOv8을 detector로 사용.
실제 이미지 시퀀스에서 Multi-Object Tracking 실행.


In [ ]:
from ultralytics import YOLO
from pathlib import Path
import glob


def run_tracking_on_sequence(model, img_paths, conf=0.3):
    """
    이미지 시퀀스에 YOLOv8 + ByteTrack 적용.
    반환: 프레임별 tracking 결과
    """
    byte_tracker = ByteTracker(thresh_high=conf, thresh_low=0.1, max_age=10, min_hits=2)
    frame_results = []

    for frame_id, img_path in enumerate(img_paths):
        result = model(img_path, conf=0.1, verbose=False)[0]  # 낮은 conf로 전체 검출

        dets = []
        if len(result.boxes):
            boxes  = result.boxes.xyxy.cpu().numpy()
            scores = result.boxes.conf.cpu().numpy()
            for box, score in zip(boxes, scores):
                dets.append([*box, score])

        tracks = byte_tracker.update(dets)
        frame_results.append({
            'frame_id': frame_id,
            'tracks': tracks,  # [[x1,y1,x2,y2,track_id], ...]
            'n_det': len(dets),
            'n_trk': len(tracks),
        })

    return frame_results


# COCO128 이미지로 테스트 (연속 프레임은 아니지만 파이프라인 검증용)
model = YOLO('runs/detect/coco128_finetune/weights/best.pt')
data_path = Path.home() / 'MyProject' / 'datasets' / 'coco128'
img_paths = sorted((data_path / 'images' / 'train2017').glob('*.jpg'))[:20]

frame_results = run_tracking_on_sequence(model, img_paths)

print('YOLOv8 + ByteTrack 연동 결과 (앞 5프레임)')
for r in frame_results[:5]:
    print(f"  Frame {r['frame_id']:2d}: det={r['n_det']:2d}, active_tracks={r['n_trk']:2d}")


In [ ]:
# ── Tracking 결과 시각화 (첫 6프레임) ──
import random
random.seed(0)
id_colors = {}

def get_id_color(tid):
    if tid not in id_colors:
        id_colors[tid] = tuple(random.randint(50, 230) for _ in range(3))
    return id_colors[tid]


fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, r in enumerate(frame_results[:6]):
    img = cv2.imread(str(img_paths[i]))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    for track in r['tracks']:
        x1, y1, x2, y2, tid = track
        x1,y1,x2,y2 = map(int, [x1,y1,x2,y2])
        tid = int(tid)
        color = get_id_color(tid)
        cv2.rectangle(img_rgb, (x1,y1), (x2,y2), color, 2)
        cv2.putText(img_rgb, f'ID:{tid}', (x1, max(y1-5,10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    axes[i].imshow(img_rgb)
    axes[i].set_title(f"Frame {r['frame_id']} | tracks={r['n_trk']}", fontsize=10)
    axes[i].axis('off')

plt.suptitle('YOLOv8 + ByteTrack — Track ID 시각화', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('tracking_visualization.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Tracking 평가 지표 — MOTA

### Detection mAP vs Tracking MOTA
| | mAP | MOTA |
|---|---|---|
| 측정 대상 | 단일 프레임 검출 정확도 | **시간에 걸친** 추적 품질 |
| FP/FN | 프레임별 독립 | 누적 카운트 |
| ID 개념 | 없음 | **ID Switch** 포함 |

### MOTA (Multiple Object Tracking Accuracy)
$$MOTA = 1 - \frac{FP + FN + ID_{SW}}{\sum_t GT_t}$$

- $ID_{SW}$: 동일 객체의 Track ID가 바뀐 횟수 (낮을수록 좋음)
- MOTA = 1.0: 완벽한 추적 / MOTA < 0: 매우 나쁨

### HOTA (Higher Order Tracking Accuracy, 2020)
$$HOTA = \sqrt{DetA \times AssA}$$

- $DetA$: Detection Accuracy (= mAP와 유사)
- $AssA$: Association Accuracy (= ID 일관성)
- MOTA보다 Detection/Association 균형 있게 평가


In [ ]:
def compute_mota(frame_results, gt_frame_results, iou_threshold=0.5):
    """
    MOTA 계산 (시뮬레이션용 간소화 버전).
    실제 MOT 벤치마크는 trackeval 라이브러리 사용.
    """
    total_fp = 0
    total_fn = 0
    total_id_sw = 0
    total_gt = 0
    prev_match = {}  # gt_id → track_id 이전 매칭

    for pred_frame, gt_frame in zip(frame_results, gt_frame_results):
        pred_tracks = pred_frame['tracks']   # [[x1,y1,x2,y2,tid], ...]
        gt_tracks   = gt_frame              # [[x1,y1,x2,y2,gt_id], ...]

        total_gt += len(gt_tracks)

        if not gt_tracks:
            total_fp += len(pred_tracks)
            continue
        if not pred_tracks:
            total_fn += len(gt_tracks)
            continue

        pred_boxes = [t[:4] for t in pred_tracks]
        gt_boxes   = [t[:4] for t in gt_tracks]
        iou_mat    = compute_iou_matrix(pred_boxes, gt_boxes)

        matched, unmatched_pred, unmatched_gt = hungarian_match(
            pred_boxes, gt_boxes, iou_threshold
        )

        total_fp += len(unmatched_pred)
        total_fn += len(unmatched_gt)

        for pi, gi in matched:
            gt_id  = int(gt_tracks[gi][4])
            trk_id = int(pred_tracks[pi][4])
            if gt_id in prev_match and prev_match[gt_id] != trk_id:
                total_id_sw += 1
            prev_match[gt_id] = trk_id

    mota = 1 - (total_fp + total_fn + total_id_sw) / max(total_gt, 1)
    return {
        'MOTA':   round(mota, 4),
        'FP':     total_fp,
        'FN':     total_fn,
        'ID_SW':  total_id_sw,
        'GT':     total_gt,
    }


# 시뮬레이션 결과로 MOTA 계산
sim_results, _ = simulate_tracking(n_frames=30)

# GT 생성 (시뮬레이션 GT)
gt_frames = []
for f in range(30):
    gt = []
    # 객체 A
    ax = 50 + f * 12
    gt.append([ax, 100, ax+80, 200, 0])
    # 객체 B (5~25프레임)
    if 5 <= f <= 25:
        bx = 400 - f * 5
        gt.append([bx, 300, bx+70, 400, 1])
    gt_frames.append(gt)

mota_result = compute_mota(sim_results, gt_frames)

print('=' * 40)
print('MOTA 평가 결과 (시뮬레이션)')
print('=' * 40)
for k, v in mota_result.items():
    print(f'  {k:6s}: {v}')
print()
print('해석:')
print(f'  MOTA > 0.5 이면 준수한 추적 성능')
print(f'  ID_SW = {mota_result["ID_SW"]} → occlusion 후 ID 유지 여부 확인')


## 7. 최종 결과 요약 및 Phase 3 연결


In [ ]:
print('=' * 60)
print('Phase 2 Tracking 파이프라인 최종 결과')
print('=' * 60)
print()
print('[구현 완료]')
print('  1. Kalman Filter   — 8차원 상태, 예측/갱신 수식 직접 구현')
print('  2. Hungarian Algo  — IoU 비용 행렬 + linear_sum_assignment')
print('  3. ByteTrack       — High/Low conf 2단계 매칭')
print('  4. MOTA 평가       — FP+FN+ID_SW 기반 지표')
print('  5. YOLOv8 연동     — Phase 1 모델 → detector로 재사용')
print()
print('[평가 지표]')
print(f'  시뮬레이션 MOTA: {mota_result["MOTA"]:.4f}')
print(f'  ID Switch:       {mota_result["ID_SW"]}')
print()
print('[Phase 1→2 재사용 확인]')
print('  IoU 구현  → hungarian_match 비용 행렬에 사용 ✓')
print('  YOLOv8    → run_tracking_on_sequence detector ✓')
print('  FP/FN 개념 → MOTA 수식으로 확장 ✓')
print()
print('[Phase 3 BEV 연결 고리]')
print('  2D Tracking    → BEV에서 3D Tracking으로 확장')
print('  카메라 좌표     → IPM으로 Bird Eye View 변환')
print('  Track ID 유지  → BEV에서도 동일 객체 추적')
